<div dir="rtl">
<h1>پیش از آموزش، داده واقعاً چیست؟</h1>
<p>درس 28 از 76 · چه چیزهایی پیش از آموزش باید دربارهٔ داده بدانیم؟ · <code dir="ltr">24-data-contract</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-04/chapter-02/24-data-contract.html">📖 بازگشت به همین درس</a></p>
<p>طول پنجره و نرخ Unknown را با قرارداد واقعی prepare_corpus بررسی کنید.</p><p>پیش‌نیاز: 21-tokenizer و23-shift؛ شمارش شناسه‌ها و فایل UTF-8.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر نویسهٔ Z فقط در انتهای اعتبارسنجی باشد، آیا باید وارد Vocabulary آموزش شود؟ آیا طول غیرصفر برای ساخت پنجره کافی است؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from mini_gpt.data import prepare_corpus
with TemporaryDirectory() as directory:
    path = Path(directory)/"corpus.txt"
    path.write_text("abcabcabcZ", encoding="utf-8")
    train_ids, valid_ids, tokenizer, info = prepare_corpus(path, train_fraction=0.6)
print("Train:", train_ids, "Validation:", valid_ids, "Metadata:", info)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع audit_ids(train_ids,valid_ids,T) دیکشنری با کلیدهای train_windows، valid_windows و unknown_rate برگرداند. تعداد پنجره‌ها len(ids)-T است. اگر هر بخش حداکثر T شناسه دارد ValueError بدهید. ID ناشناخته صفر است؛ T در این تمرین مثبت است.</p>
</div>

In [ ]:
def audit_ids(train_ids, valid_ids, T):
    # TODO: audit the actual encoded input, not just the original text
    return None

In [ ]:
def test_exercise():
    result = audit_ids(train_ids, valid_ids, 2)
    if result is None:
        return False
    assert result == {"train_windows": 4, "valid_windows": 2, "unknown_rate": 0.25}
    assert result["unknown_rate"] == info["validation_unknown_rate"]
    assert "Z" not in tokenizer.token_to_id
    assert audit_ids([1, 2, 3], [0, 0, 1], 1)["unknown_rate"] == 2/3
    try:
        audit_ids([1, 2], [1, 2, 3], 2)
    except ValueError:
        pass
    else:
        raise AssertionError("A final target must exist in both splits")
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط یک نویسه از محتوای فایل موقت را عوض کنید. اثر انگشت باید عوض شود؛ این تغییر چیزی دربارهٔ بهتر یا بدترشدن کیفیت متن نمی‌گوید.</p>
</div>

In [ ]:
import hashlib
for text in ["abcabcabcZ", "abcabcabcY"]:
    print(text, hashlib.sha256(text.encode("utf-8")).hexdigest())

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>ساخت Vocabulary از کل متن، Unknown اعتبارسنجی را پنهان می‌کند. تابع train_tokenizer(text,fraction) فقط برش آموزش را به CharacterTokenizer بدهد؛ fraction معتبر و برش آموزش غیرخالی است.</p>
</div>

In [ ]:
from mini_gpt.tokenizer import CharacterTokenizer
whole_text = "abcabcabcZ"
wrong_tokenizer = CharacterTokenizer.from_text(whole_text)
print("Leaked Z ID:", wrong_tokenizer.encode("Z"))
assert wrong_tokenizer.encode("Z") != [0]

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def train_tokenizer(text, fraction):
    # TODO: build vocabulary only from the training prefix
    return None

In [ ]:
def test_repair():
    result = train_tokenizer(whole_text, 0.6)
    if result is None:
        return False
    assert result.encode("Z") == [0]
    assert result.id_to_token == tokenizer.id_to_token
    assert train_tokenizer("aaaaZZ", 2/3).encode("Z") == [0]
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>این آزمایش prepare_corpus واقعی را با فایل موقت اجرا می‌کند. فایل data/sample.txt و داده‌های کاربر را تغییر نمی‌دهد؛ اطلاعات خروجی همان اطلاعات ذخیره‌شده در آموزش پروژه است.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا کاهش نرخ Unknown با دیدن کل داده، لزوماً به معنی بهترشدن آزمایش نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-04/chapter-02/24-data-contract.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/24-data-contract.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>